# Transforms and images

Reuse a Transform on uint8, uint16 and float images, carry alpha, and control intents and threads.

In [1]:
import numpy as np

import lcms2

A small seeded gradient plus noise stands in for a photo: an (H, W, 3) uint8 sRGB array.

In [2]:
rng = np.random.default_rng(0)
ramp = np.linspace(0, 255, 64)
img = np.stack(np.broadcast_arrays(ramp[None, :], ramp[:, None], 128.0), axis=-1)
img = (img + rng.normal(0, 4, img.shape)).clip(0, 255).astype(np.uint8)
print(img.shape, img.dtype)

(64, 64, 3) uint8


A `Transform` is built once and called many times. The pixel formats are resolved from each input and
cached, so the same object takes uint8, uint16 and float arrays.

In [3]:
to_p3 = lcms2.Transform("sRGB", "Display P3")
print(to_p3(img)[0, -1], to_p3(img).dtype)
print(to_p3(img.astype(np.uint16) * 257)[0, -1], to_p3(img.astype(np.uint16) * 257).dtype)
print(to_p3(img.astype(np.float32) / 255)[0, -1].round(4))

[232  51 122] uint8
[59653 13002 31308] uint16
[0.9103 0.1984 0.4777]


An extra last channel is alpha and is copied through unchanged. `out=` writes into an array you own.

In [4]:
rgba = np.concatenate([img, np.full(img.shape[:2] + (1,), 200, np.uint8)], axis=-1)
result = to_p3(rgba)
print(result.shape, result[0, 0], rgba[0, 0])
buffer = np.empty_like(img)
to_p3(img, out=buffer)
print(np.array_equal(buffer, to_p3(img)))

(64, 64, 4) [  0   0 124 200] [  0   0 130 200]
True


`intent` takes a name or an `lcms2.Intent`. Matrix/TRC profiles render every intent alike; intents matter
for CLUT profiles such as printers. `bpc=True` scales the source black point to the target's, shown here
with a profile whose black is lifted to 5% of the signal.

In [5]:
print([intent.name for intent in lcms2.Intent][:4])
lifted = lcms2.Profile.rgb("D50", gamma=lcms2.ToneCurve.tabulated(np.linspace(0.05, 0.9, 32)))
print("black point", lifted.black_point("relative").round(3))
black = np.array([0, 0, 0], np.uint8)
for bpc in (False, True):
    print("bpc", bpc, lcms2.convert(black, lifted, "sRGB", intent="relative", bpc=bpc))

['PERCEPTUAL', 'RELATIVE_COLORIMETRIC', 'SATURATION', 'ABSOLUTE_COLORIMETRIC']
black point [0.048 0.05  0.041]
bpc False [63 63 63]
bpc True [0 0 0]


A list of profiles is one multi-profile transform, here through a contrast-boosting abstract profile.

In [6]:
contrast = lcms2.Profile.abstract(contrast=1.3)
chain = lcms2.Transform(["sRGB", contrast, "Display P3"])
print(chain(np.array([[100, 120, 140]], np.uint8)))
print(to_p3(np.array([[100, 120, 140]], np.uint8)))

[[142 158 177]]
[[104 119 138]]


Explicit LittleCMS formats replace the automatic ones; `lcms2.FORMATS` has all 178.

In [7]:
t16 = lcms2.Transform("sRGB", "RGB_8", "Display P3", "RGB_16")
print(t16.input_format, t16.output_format)
print(t16(img)[0, -1])
print(list(lcms2.FORMATS)[:5])

RGB_8 RGB_16
[59654 12999 31310]
['GRAY_8', 'GRAY_8_REV', 'GRAY_16', 'GRAY_16_REV', 'GRAY_16_SE']


Large inputs are split across a shared thread pool (262,144 pixels or more). The result does not depend
on the thread count; `set_threads(1)` turns the pool off, for example inside your own pool.

In [8]:
big = np.tile(img, (64, 64, 1))
print(big.shape)
default = lcms2.get_threads()
many = to_p3(big)
lcms2.set_threads(1)
single = to_p3(big)
lcms2.set_threads(default)
print(np.array_equal(many, single), lcms2.get_threads() == default)

(4096, 4096, 3)


True True
